# 01. PubChem 기반 데이터 수집

> 서울시립대 교육 · 한국화학연구원(KRICT)

이 실습에서는 **PubChem REST API (PUG-REST)** 를 이용해 URL 요청만으로
화합물(Chemical) 정보와 생리활성(BioAssay) 데이터를 직접 수집한다.

## 학습 목표
1. PubChem REST API(PUG-REST)의 **URL 구조**를 이해한다.
2. 화합물 이름/CID로 **속성·별명·설명·구조**를 수집한다. *(예: Aspirin, CID 2244)*
3. BioAssay(AID)로부터 **화합물과 활성(Active/Inactive) 데이터**를 수집한다. *(예: AID 651580, HIF-2a 저해제)*
4. 수집한 데이터를 **pandas DataFrame** 으로 정리하고 CSV로 저장한다.

## PUG-REST 란?
PubChem의 **P**ower **U**ser **G**ateway - **REST** 인터페이스로,
웹 주소(URL)를 조합해 GET 요청을 보내면 JSON/CSV/PNG 형태로 데이터를 돌려준다.
별도의 라이브러리 없이 `requests` 만으로 사용할 수 있다는 것이 장점이다.

```
https://pubchem.ncbi.nlm.nih.gov/rest/pug / <입력> / <조회 대상> / <출력형식>
                     (base)                  (input)   (operation)    (output)
```

- 공식 문서: https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest

## 0. 환경 설정

실습에 필요한 라이브러리를 불러온다. (Google Colab에는 모두 기본 설치되어 있음)
- `requests` : REST API에 HTTP 요청을 보내는 라이브러리
- `pandas`   : 수집한 데이터를 표(DataFrame) 형태로 다루기 위한 라이브러리
- `time`     : PubChem 서버 부하 방지를 위한 요청 간 대기

In [ ]:
import requests          # REST API(URL) 요청
import pandas as pd      # 데이터프레임 처리
import time              # 요청 간 간격 조절
from io import StringIO  # CSV 문자열을 DataFrame으로 변환

# PUG-REST 서비스의 기본 주소(base URL). 이후 모든 요청은 이 주소 뒤에 경로를 붙여 만든다.
BASE = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'

print('환경 설정 완료:', requests.__version__, pd.__version__)

---
# 1. PubChem Chemical(Compound) 데이터 수집

> **[실습] REST API 기반 (url) PubChem Chemical 데이터 수집**

PubChem의 화합물 페이지(예: [Aspirin, CID 2244](https://pubchem.ncbi.nlm.nih.gov/compound/2244))
에 정리된 정보(분자식, 분자량, 별명, 구조, 설명 등)를 API로 그대로 가져와 본다.

### Compound 요청 URL 구조
```
{BASE}/compound/{입력방식}/{값}/{조회대상}/{출력형식}
```
| 구성 | 예시 | 설명 |
|------|------|------|
| 입력방식(input) | `name`, `cid`, `smiles` | 무엇으로 찾을지 |
| 값 | `aspirin`, `2244` | 검색 값 |
| 조회대상(operation) | `property/...`, `synonyms`, `description` | 무엇을 가져올지 |
| 출력형식(output) | `JSON`, `CSV`, `PNG`, `TXT` | 어떤 형식으로 받을지 |

## 1-1. 화합물 이름 → CID 조회

CID(Compound ID)는 PubChem이 화합물마다 부여하는 고유 번호다.
먼저 화합물 **이름(name)** 으로 CID를 찾아본다. (Aspirin → 2244)

In [ ]:
compound_name = 'aspirin'

# name → CID : /compound/name/{이름}/cids/JSON
url = f'{BASE}/compound/name/{compound_name}/cids/JSON'
print('요청 URL:', url)

res = requests.get(url)      # GET 요청 전송
res.raise_for_status()       # 오류(4xx/5xx)면 예외 발생시켜 확인
data = res.json()            # 응답(JSON)을 파이썬 dict로 변환

cid = data['IdentifierList']['CID'][0]   # 첫 번째 CID 사용
print(f'{compound_name} 의 CID =', cid)

## 1-2. 화합물 속성(Property) 수집

슬라이드의 **Molecular Formula / Molecular Weight** 등에 해당하는 값들을 가져온다.
여러 속성을 쉼표로 이어서 한 번에 요청할 수 있다.

- `MolecularFormula` : 분자식 (예: C9H8O4)
- `MolecularWeight`  : 분자량 (예: 180.16 g/mol)
- `SMILES`           : 구조를 나타내는 문자열(SMILES)
- `IUPACName`        : IUPAC 명명법 이름
- `XLogP`            : 지용성 지표(logP)

In [ ]:
# 가져올 속성 목록
props = 'MolecularFormula,MolecularWeight,SMILES,IUPACName,XLogP'

# cid → property : /compound/cid/{cid}/property/{속성목록}/JSON
url = f'{BASE}/compound/cid/{cid}/property/{props}/JSON'
res = requests.get(url)
res.raise_for_status()

# 응답 안의 속성 딕셔너리를 꺼낸다 (PropertyTable > Properties > 첫 항목)
prop = res.json()['PropertyTable']['Properties'][0]
prop

## 1-3. Synonyms(별명) 수집

슬라이드의 **Synonyms** 항목에 해당한다.
(예: aspirin, ACETYLSALICYLIC ACID, 50-78-2, 2-Acetoxybenzoic acid ...)
동의어는 수백 개가 넘을 수 있으므로 앞부분 몇 개만 확인한다.

In [ ]:
# cid → synonyms : /compound/cid/{cid}/synonyms/JSON
url = f'{BASE}/compound/cid/{cid}/synonyms/JSON'
res = requests.get(url)
res.raise_for_status()

synonyms = res.json()['InformationList']['Information'][0]['Synonym']
print('총 별명 개수:', len(synonyms))
synonyms[:5]     # 대표 별명 5개만 미리보기

## 1-4. Description(설명) 수집

슬라이드 하단의 **Description** 에 해당한다.
화합물에 대한 서술형 설명과 출처를 가져온다.

In [ ]:
# cid → description : /compound/cid/{cid}/description/JSON
url = f'{BASE}/compound/cid/{cid}/description/JSON'
res = requests.get(url)
res.raise_for_status()

# 설명이 담긴 항목만 골라 출력 (Description 키가 있는 항목)
for info in res.json()['InformationList']['Information']:
    if 'Description' in info:
        print('[출처]', info.get('DescriptionSourceName', '-'))
        print(info['Description'])
        print('-' * 80)

## 1-5. 구조 이미지(2D) 확인

출력형식을 `PNG` 로 지정하면 화합물의 2D 구조 그림을 이미지로 받을 수 있다.
슬라이드의 **Structure(2D)** 에 해당한다.

In [ ]:
from IPython.display import Image

# cid → 구조 이미지 : /compound/cid/{cid}/PNG
url = f'{BASE}/compound/cid/{cid}/PNG'
print('이미지 URL:', url)

Image(url=url)   # 노트북에 2D 구조 이미지 표시

## 1-6. 여러 화합물 한 번에 수집하기 (함수화)

실무에서는 화합물 하나가 아니라 **여러 개**를 반복 수집한다.
위 과정을 함수로 묶어 여러 물질을 한꺼번에 표(DataFrame)로 만들어 본다.

> 서버 부하를 줄이기 위해 요청 사이에 `time.sleep()` 으로 잠깐 쉬어 준다.
> (PubChem 권장: 초당 5회 이하)

In [ ]:
def fetch_compound(name):
    """화합물 이름을 받아 주요 속성을 dict로 반환한다."""
    props = 'MolecularFormula,MolecularWeight,SMILES,IUPACName'
    url = f'{BASE}/compound/name/{name}/property/{props}/JSON'
    res = requests.get(url)
    if res.status_code != 200:      # 검색 실패 시 건너뜀
        print(f'  [실패] {name}: HTTP {res.status_code}')
        return None
    row = res.json()['PropertyTable']['Properties'][0]
    row['Name'] = name              # 원래 검색어도 함께 기록
    return row

# 여러 약물 이름을 반복 수집
names = ['aspirin', 'ibuprofen', 'caffeine', 'acetaminophen']
rows = []
for name in names:
    print('수집 중:', name)
    row = fetch_compound(name)
    if row:
        rows.append(row)
    time.sleep(0.2)                 # 요청 간 0.2초 대기

df_compounds = pd.DataFrame(rows)
df_compounds

---
# 2. PubChem BioAssay 데이터 수집

> **[실습] REST API 기반 (url) PubChem BioAssay 데이터 수집**
>
> PubChem REST API BioAssay 로부터 chemical 및 활성 데이터 수집

**BioAssay** 는 화합물의 생리활성 실험 결과를 모아 둔 데이터로,
각 실험은 **AID(Assay ID)** 라는 고유 번호를 가진다.

이번 실습 대상은 슬라이드와 동일하게
[**AID 651580** — *Single concentration confirmation of uHTS identification of HIF-2a Inhibitors*](https://pubchem.ncbi.nlm.nih.gov/bioassay/651580)
이다. (Target: HIF-2a / endothelial PAS domain-containing protein 1)

### Assay 요청 URL 구조
```
{BASE}/assay/aid/{AID}/{조회대상}/{출력형식}
```
- `description` : 어세이 개요(제목, 타겟, 출처 등)
- `cids`        : 실험된 화합물 CID 목록 (전체/Active/Inactive)
- `CSV`         : 화합물별 활성 결과 데이터 표

In [ ]:
AID = 651580   # 실습 대상 BioAssay ID (HIF-2a 저해제)

## 2-1. Assay 개요(Description) 수집

슬라이드의 **Protein Target / Source / BioAssay Type / Description** 등
어세이의 기본 정보를 가져온다.

In [ ]:
# aid → description : /assay/aid/{AID}/description/JSON
url = f'{BASE}/assay/aid/{AID}/description/JSON'
res = requests.get(url)
res.raise_for_status()

assay = res.json()['PC_AssayContainer'][0]['assay']['descr']

print('AID   :', assay['aid']['id'])
print('제목  :', assay['name'])
# 어세이 설명은 여러 줄의 리스트로 제공된다
print('설명  :')
for line in assay.get('description', [])[:5]:
    print('  ', line)

## 2-2. 화합물별 활성 데이터 표(CSV) 수집

출력형식을 `CSV` 로 지정하면 화합물별 활성 결과를 표로 받을 수 있다.
핵심 컬럼은 다음과 같다.
- `PUBCHEM_CID`               : 화합물 ID
- `PUBCHEM_ACTIVITY_OUTCOME`  : 활성 판정 (Active / Inactive)
- `PUBCHEM_ACTIVITY_SCORE`    : 활성 점수

슬라이드의 **Tested Substances: All(1,759) / Active(982) / Inactive(777)** 에 해당하는
원본 데이터다.

In [ ]:
# aid → 활성 데이터 표 : /assay/aid/{AID}/CSV
url = f'{BASE}/assay/aid/{AID}/CSV'
print('요청 URL:', url)

res = requests.get(url)
res.raise_for_status()

# 응답 본문(CSV 문자열)을 pandas DataFrame으로 변환
df_assay = pd.read_csv(StringIO(res.text))
print('데이터 크기(행, 열):', df_assay.shape)
df_assay.head()

In [ ]:
# 활성 판정 분포 확인 (슬라이드의 Active/Inactive 개수와 비교)
df_assay['PUBCHEM_ACTIVITY_OUTCOME'].value_counts()

## 2-3. Active / Inactive 화합물 CID 목록

`cids` 조회에 `cids_type` 옵션을 주면 활성/비활성 화합물만 따로 받을 수 있다.
- `cids_type=active`   : Active 화합물
- `cids_type=inactive` : Inactive 화합물

In [ ]:
def get_cids(aid, cids_type):
    """어세이(AID)에서 특정 유형(active/inactive)의 CID 목록을 반환한다."""
    url = f'{BASE}/assay/aid/{aid}/cids/JSON?cids_type={cids_type}'
    res = requests.get(url)
    res.raise_for_status()
    return res.json()['InformationList']['Information'][0].get('CID', [])

active_cids   = get_cids(AID, 'active')
inactive_cids = get_cids(AID, 'inactive')

print('Active   화합물 수:', len(active_cids))
print('Inactive 화합물 수:', len(inactive_cids))
print('예시 Active CID   :', active_cids[:5])

## 2-4. 활성 데이터에 화합물 구조(SMILES) 결합

활성 데이터(`df_assay`)에는 CID만 있고 구조 정보가 없다.
머신러닝에 쓰려면 각 CID의 **SMILES**(구조 문자열)가 필요하다.
CID 목록으로 SMILES를 한 번에 받아 활성 데이터와 합친다.

> 실습 시간 단축을 위해 앞쪽 일부(예: 200개)만 사용한다.
> 전체를 쓰려면 `SAMPLE_N` 값을 늘리면 된다.

In [ ]:
# 활성 데이터에서 유효한 CID만 추출 (결측 제거 후 정수형 변환)
cid_series = df_assay['PUBCHEM_CID'].dropna().astype(int)

SAMPLE_N = 200                       # 실습용 샘플 개수
target_cids = cid_series.unique()[:SAMPLE_N].tolist()
print('SMILES를 수집할 CID 개수:', len(target_cids))

def fetch_smiles(cids, chunk=100):
    """CID 리스트를 100개씩 나누어 SMILES를 조회하고 DataFrame으로 반환한다."""
    frames = []
    for i in range(0, len(cids), chunk):
        part = cids[i:i + chunk]
        cid_str = ','.join(map(str, part))   # 여러 CID를 쉼표로 연결
        url = f'{BASE}/compound/cid/{cid_str}/property/SMILES/CSV'
        res = requests.get(url)
        res.raise_for_status()
        frames.append(pd.read_csv(StringIO(res.text)))
        time.sleep(0.2)                      # 요청 간 대기
    return pd.concat(frames, ignore_index=True)

df_smiles = fetch_smiles(target_cids)
df_smiles.head()

In [ ]:
# 활성 데이터(CID, 활성판정)와 SMILES를 CID 기준으로 병합
df_activity = df_assay[['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME']].dropna()
df_activity['PUBCHEM_CID'] = df_activity['PUBCHEM_CID'].astype(int)

df_dataset = df_smiles.merge(
    df_activity,
    left_on='CID', right_on='PUBCHEM_CID', how='inner'
)

# 필요한 컬럼만 정리하고 이름을 알기 쉽게 변경
df_dataset = df_dataset[['CID', 'SMILES', 'PUBCHEM_ACTIVITY_OUTCOME']]
df_dataset = df_dataset.rename(columns={
    'PUBCHEM_ACTIVITY_OUTCOME': 'Activity',
})

print('최종 데이터셋 크기:', df_dataset.shape)
print(df_dataset['Activity'].value_counts())
df_dataset.head()

## 2-5. 데이터 저장

수집한 데이터셋을 CSV 파일로 저장한다.
이 파일은 다음 강의 **`02_preprocessing_rdkit.ipynb`** 에서 전처리에 사용한다.

In [ ]:
out_path = 'hif2a_aid651580_dataset.csv'
df_dataset.to_csv(out_path, index=False)
print('저장 완료:', out_path)

# (Colab) 파일 내려받기 - 필요 시 주석 해제
# from google.colab import files
# files.download(out_path)

---
## 정리

이번 실습에서 배운 내용:

| 구분 | 사용한 API 경로 | 얻은 데이터 |
|------|----------------|------------|
| Chemical | `/compound/name|cid/.../property\|synonyms\|description\|PNG` | 분자식·분자량·SMILES·별명·설명·구조 |
| BioAssay | `/assay/aid/{AID}/description\|cids\|CSV` | 어세이 개요·활성 판정·화합물 목록 |

핵심 포인트
- PUG-REST는 **URL만 조합**하면 별도 로그인/키 없이 데이터를 받을 수 있다.
- 출력형식(JSON/CSV/PNG)을 바꿔 상황에 맞게 활용한다.
- 대량 요청 시 **CID를 묶어서** 요청하고 **`time.sleep()`** 으로 서버 부하를 줄인다.
- 수집 결과는 **DataFrame → CSV** 로 저장해 이후 단계로 넘긴다.

**➡️ 다음 강의:** `02_preprocessing_rdkit.ipynb` — 수집한 SMILES를 RDKit으로 전처리하고 분자 특성(descriptor/fingerprint)을 생성한다.